Someone asked in the forum how many days it takes for an agent's
score to become stable, and the honest answer turned out to be that days
are not the unit the question needs. This notebook is the measurement I
use instead, with the code and four real sampled series behind it.

**The short version.** A rating moves per EPISODE, not per hour, and the
episode rate varies by a factor of ten between agents, so the same
"two days" can mean 17 episodes or 170. Sample the pair (score, episodes
played), take the drift in points per episode, and treat the number as
readable when that drift is small and its sign has flipped at least
once. And a rating never becomes stable in the strict sense, because it
measures you against a field that keeps changing: one of my agents fell
182 points over 135 episodes without a single byte of it changing.

Nothing here is about which agent to build. It is only about when a
number on the leaderboard is worth reading.

<a id="s1"></a>
## 1. Why the clock is the wrong instrument

Four of my own submissions, each sampled repeatedly while it played.
What varies is not how fast the rating moves, it is how fast the ladder
feeds the agent games at all.

In [1]:
import matplotlib.pyplot as plt

PROBES = [('A, warming up', 11, 88, 12.3), ('B, byte-identical', 22, 55, 16.4), ('C, settled', 36, 13, 9.7), ('D, quiet', 10, 3, 4.5)]

labels = [p[0] for p in PROBES]
rate = [p[2] / p[3] for p in PROBES]          # episodes per hour
y = list(range(len(labels)))[::-1]

fig, ax = plt.subplots(figsize=(8.6, 3.0))
ax.barh(y, rate, height=0.6, color="#2f6fb2")
for yy, r, p in zip(y, rate, PROBES):
    ax.annotate(f"{r:.1f}/h   ({p[2]} episodes in {p[3]:.1f} h)",
                (r, yy), xytext=(6, 0), textcoords="offset points",
                va="center", fontsize=9, color="#40484f")
ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=9)
ax.set_xlim(0, 11)
ax.set_xlabel("episodes played per hour")
ax.set_title("the same wall clock buys very different amounts of evidence",
             fontsize=10)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

A factor of ten between the fastest and the slowest. Converted to
the unit the original question used, one day is about 17 episodes for
the slow agent and about 170 for the fast one, so "wait two days" is not
a specification of anything.

The rate is not constant within one agent either. A newly published
submission gets a burst while the ladder places it, then settles, and
publishing a second submission slows the first one down again.

<a id="s2"></a>
## 2. Three real trajectories, plotted against episodes

Same three agents, now with the x axis in the unit that carries the
information. Reading them left to right is the whole method.

In [2]:
import matplotlib.pyplot as plt

WARMING = [(13, 1809.8), (16, 1895.2), (32, 2394.2), (96, 2574.3), (100, 2583.7), (101, 2578.9)]
FALLING = [(80, 2913.7), (89, 2892.9), (91, 2884.0), (92, 2880.3), (93, 2886.4), (129, 2750.6), (134, 2735.5), (135, 2731.9)]
SETTLED = [(80, 2036.6), (88, 2027.7), (89, 2023.2), (92, 2018.1), (93, 2022.6)]

fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.4))
series = [("A: still warming up", WARMING, "#c2571f",
           "+12.0 pts/episode"),
          ("B: falling, byte-identical", FALLING, "#b5486b",
           "-2.11 pts/episode"),
          ("C: settled", SETTLED, "#14724d", "-0.70 pts/episode")]
for ax, (title, s, colour, note) in zip(axes, series):
    xs = [p[0] for p in s]; ys = [p[1] for p in s]
    ax.plot(xs, ys, "-o", ms=4, color=colour)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("episodes played")
    ax.annotate(note, (0.5, 0.06), xycoords="axes fraction", ha="center",
                fontsize=9, color=colour, weight="bold")
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
axes[0].set_ylabel("leaderboard rating")
plt.tight_layout(); plt.show()

**A is not readable.** It gained 769 points across 88 episodes and
was still climbing when I stopped sampling. Any number taken from that
curve describes where the placement process had got to, not the agent.

**B is the one worth staring at.** That agent did not change: same file,
same bytes, from the first sample to the last. It lost 182 points over
55 episodes anyway, at a steady 2.11 points per episode.

**C is what settled looks like:** small moves, and the sign changes
rather than continuing in one direction.

<a id="s3"></a>
## 3. The measure, and the two places it goes wrong

The rule itself is four lines. What matters is the filtering around it.

In [3]:
def convergence(samples, threshold=1.0):
    """samples: list of (episodes_completed, score) for ONE submission,
    in time order. Returns the per-episode drift, the number of sign
    flips, and whether the rating can be read yet."""
    steps = [(s2 - s1) / (n2 - n1)
             for (n1, s1), (n2, s2) in zip(samples, samples[1:])
             if n2 > n1]                       # <- the important filter
    if not steps:
        return None, 0, False
    drift = sum(steps) / len(steps)
    flips = sum(1 for a, b in zip(steps, steps[1:]) if a * b < 0)
    return drift, flips, abs(drift) <= threshold and flips >= 1


for name, s in (("A, warming up", WARMING),
                ("B, byte-identical", FALLING),
                ("C, settled", SETTLED)):
    d, f, ok = convergence(s)
    print(f"{name:20s} drift {d:+6.2f} pts/episode   flips {f}   "
          f"{'READABLE' if ok else 'not readable yet'}")

**Trap one: the samples that carry no information.** The
leaderboard updates in batches, so most probes come back with a new
timestamp and the same episode count. Of the 36 probes I took on agent
C, **31 showed no new episode**, and of the 22 on agent B, 14 did. Those
samples are not evidence of stability, they are evidence of nothing, and
averaging them in pulls any drift estimate towards zero. That is what
the `if n2 > n1` filter is for, and dropping it is the difference
between "settled" and "still moving" on the same data.

**Trap two: mixing submissions in one series.** If the sampler tracks
"my best submission" rather than a fixed submission id, it will
eventually compute a delta between two unrelated ratings and divide it
by a difference of unrelated episode counts. The first version of mine
did exactly that and reported a drift of +95 points per episode, which
is not a quantity that exists. Key the series by submission id.

**Why the sign flip is in the rule.** A drift near zero can mean the
rating has settled, or that it happens to be passing through zero on
its way somewhere, and a direction change separates the two cheaply.
Agent A is rejected by the drift alone, at 12 points per episode, but a
slower version of the same climb would show a small average drift while
never turning around, and that is the case the flip requirement is
there to catch.

<a id="s4"></a>
## 4. The part that surprised me: settled is not stable

Agent B is the interesting case, and it is worth being precise about
what it shows. The agent was byte-identical throughout. The 182 points
it lost were not a property of the agent at all.

A rating is a measurement of you **against the field of that moment**,
and the field turns over: people publish, agents get replaced, the
matchmaking hands you different opponents. So the same agent measured a
week apart is measured against two different populations, and the two
numbers are not comparable even though they look like the same
quantity.

Three consequences I now treat as rules:

* **Two readings of the same agent taken at different times compare
  nothing**, including my own agent against its own earlier peak.
* **A retired submission's rating is frozen, not final.** It stops where
  it stopped when it left the window, which flatters agents that played
  briefly. An agent that played 10 hours and one that played 200 are not
  on the same scale.
* **To compare two of my own agents I have to field both and read them
  together**, in the same window against the same moving field.

In [4]:
import matplotlib.pyplot as plt

xs = [p[0] for p in FALLING]; ys = [p[1] for p in FALLING]
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(xs, ys, "-o", ms=5, color="#b5486b")
ax.axhline(ys[0], color="#9aa3ad", ls="--", lw=1)
ax.annotate("first reading, 2,913.7", (xs[0], ys[0]), xytext=(12, 9),
            textcoords="offset points", fontsize=9, color="#40484f")
ax.annotate("last reading, 2,731.9", (xs[-1], ys[-1]), xytext=(-10, 13),
            textcoords="offset points", ha="right", fontsize=9,
            color="#40484f")
ax.annotate("182 points, and not one byte\nof the agent changed",
            (0.24, 0.12), xycoords="axes fraction", ha="center",
            fontsize=10, color="#b5486b", weight="bold")
ax.margins(y=0.18)
ax.set_xlabel("episodes played")
ax.set_ylabel("leaderboard rating")
ax.set_title("the same file, measured against a field that moved",
             fontsize=10)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

<a id="s5"></a>
## 5. The whole sampler

Two pieces: a probe that reads one submission's score and its completed
episode count, and a loop that keeps probing until enough NEW episodes
have accumulated to say something. This is the version I actually run,
including the three parsing details that took a couple of passes to get
right, each flagged in a comment.

In [5]:
import csv
import io
import subprocess
import time


def _csv(args):
    """Run a kaggle CLI command with --csv and parse it properly."""
    out = subprocess.run(["kaggle", *args, "--csv"],
                         capture_output=True, text=True, check=True).stdout
    lines = out.splitlines()
    # (1) some commands print a "Next Page Token = ..." preamble before
    #     the CSV, so find the header rather than assuming line 0.
    start = next((i for i, l in enumerate(lines)
                  if "," in l and not l.startswith("Next Page")), None)
    if start is None:
        return []
    # (2) use the csv module: submission descriptions are quoted and
    #     contain commas, so splitting on "," silently misaligns fields.
    return list(csv.DictReader(io.StringIO("\n".join(lines[start:]))))


def probe(submission_id, competition="kaggriculture"):
    """(ladder episodes completed, public score) for one submission."""
    subs = _csv(["competitions", "submissions", competition])
    row = next((r for r in subs
                if str(r.get("ref")) == str(submission_id)), None)
    score = float(row.get("publicScore") or 0) if row else 0.0

    eps = _csv(["competitions", "episodes", str(submission_id)])
    # (3) PUBLIC only: every submission also plays one VALIDATION
    #     episode against itself, which is not a rated game. It is a
    #     constant +1, so it cancels in a difference and does not change
    #     the drift, but it does change "how many games has this played".
    #     The same filter drops the usage-hint line the CLI appends.
    played = sum(1 for e in eps
                 if "COMPLET" in str(e.get("state", ""))
                 and "PUBLIC" in str(e.get("type", "")))
    return played, score


def sample_until(submission_id, min_new_episodes=8, every=1800,
                 max_probes=12):
    """Probe on a timer until enough NEW episodes have been played."""
    samples, start = [], None
    for _ in range(max_probes):
        n, score = probe(submission_id)
        samples.append((n, score))
        start = n if start is None else start
        drift, flips, readable = convergence(samples)
        print(f"episodes {n:4d}  score {score:8.1f}  "
              f"drift {drift if drift is None else round(drift, 2)}")
        if n - start >= min_new_episodes and readable:
            break
        time.sleep(every)
    return samples

On the interval: pick it so that consecutive probes usually differ
by at least one episode, otherwise almost every sample is discarded by
the filter and the loop just burns API calls. At the rates in section 1
that is somewhere between fifteen minutes and an hour, and measuring
your own rate once beats guessing it.

The `check=True` is deliberate. A CLI call that fails for an unrelated
reason, an expired token or a rate limit, otherwise returns an empty
string, which parses to zero episodes and a score of 0.0 and quietly
enters the series as a data point.

<a id="s6"></a>
## 6. What I actually do with it

I run the sampler before reading any score, and I quote the threshold
with the verdict, because "settled at 1 point per episode" and "settled
at 5" are different claims. When a run comes back not readable, the
answer to "is this agent better" is "not enough evidence yet", which is
a different statement from "no difference" and gets treated differently.

And when I want to compare two of my own agents, I field both and read
them in the same window, since that is the only comparison the ladder
supports.

If you sample your own submissions this way I would be curious what
episode rates you see, and whether the ten-fold spread in section 1 is
typical or particular to how I was publishing.